In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('train.txt',sep = ';',header=None,names=['text','emotions'])

In [ ]:
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [ ]:
df.shape

(16000, 2)

In [ ]:
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [ ]:
df['emotions'].unique()
unique_emotions = df['emotions'].unique()
emotions_map= {}

i = 0
for emo in unique_emotions:
  emotions_map[emo] = i
  i += 1

df['emotions'] = df['emotions'].map(emotions_map)

In [ ]:
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


**1.CONVERT TO LOWERCASE**

In [ ]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [ ]:
df['text'].head()

,text
0,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...
4,i am feeling grouchy


**2 . REMOVE PUNCTUATIONS**

In [ ]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [ ]:
df['text'] = df['text'].apply(remove_punc)

In [ ]:
df['text'].head()

,text
0,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...
4,i am feeling grouchy


**3 . REMOVE NUMBERS**

In [ ]:
def remove_num(txt):
  word = ""
  for i in txt:
    if not i.isdigit():
      word += i
  return word

In [ ]:
df['text'] = df['text'].apply(remove_num)
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


**4 . REMOVE EMOJIS OR SPECIAL CHARACTERS**

In [ ]:
def remove_emojis(txt):
  word = ""
  for i in txt:
    if ord(i) < 128:
            word += i
  return word

In [ ]:
df['text'] = df['text'].apply(remove_emojis)
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


5 .REMOVE STOPWORDS

In [ ]:
import nltk

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [ ]:
def remove_stopwords(txt):
  words = txt.split()
  cleaned = []

  for i in words:
    if not i in stop_words:
      cleaned.append(i)
      cleaned.append(' ')

  return ''.join(cleaned)

In [ ]:
df['text'] = df['text'].apply(remove_stopwords)
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake '

In [ ]:
df.head()

,text,emotions
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


**SPLITTING INTO TRAINING AND TESTING**

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['emotions']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

**TF-IDF VECTORIZER**

1 . NAIVE BAYES

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train_tfidf,y_train)
y_pred_nb = model_nb.predict(X_test_tfidf)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred_nb)

0.6609375

2 . LOGISTIC REGRESSION

In [91]:
from sklearn.linear_model import LogisticRegressionCV

model_logr = LogisticRegressionCV()
model_logr.fit(X_train_tfidf,y_train)
y_pred_logr = model_logr.predict(X_test_tfidf)

accuracy_score(y_test,y_pred_logr)

0.8784375

**COUNT VECTORIZER**

1 . NAIVE BAYES

In [94]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

model_nb_bow = MultinomialNB()
model_nb.fit(X_train_bow,y_train)
y_pred_nb_bow = model_nb.predict(X_test_bow)

accuracy_score(y_test,y_pred_nb_bow)

0.768125

2 . LOGISTIC REGRESSION

In [96]:
model_logr_bow = LogisticRegressionCV()
model_logr_bow.fit(X_train_bow,y_train)
y_pred_logr_bow = model_logr_bow.predict(X_test_bow)

accuracy_score(y_test,y_pred_logr_bow)

0.89125